# Named Entity Recognition (NER) for News Articles



---
## Table of Contents
1. Installation
2. Imports & Global Config
3. Data Loading
4. Preprocessing
5. Feature Engineering
6. Model Training
7. Evaluation
8. Inference
9. Appendix — Zero-shot Demo


In [17]:
!pip install -q transformers datasets seqeval torch pandas scikit-learn spacy
!python -m spacy download en_core_web_sm -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 80.3 MB/s eta 0:00:0000:010:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


## 2. Imports & Global Config


In [18]:
import warnings, os
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    pipeline,
)
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score
import spacy

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE           = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_CHECKPOINT = 'distilbert-base-cased'
MAX_LENGTH       = 128
BATCH_SIZE       = 16
NUM_EPOCHS       = 3
LEARNING_RATE    = 2e-5
OUTPUT_DIR       = './ner_model'

print(f'Device : {DEVICE}')
print(f'Model  : {MODEL_CHECKPOINT}')


Device : cuda
Model  : distilbert-base-cased


## 3. Data Loading

We use **`unimelb-nlp/wikiann`** (English subset) / a fully **parquet-native** NER dataset. It uses the same IOB2 BIO scheme as CoNLL-2003: `O`, `B-PER`, `I-PER`, `B-ORG`, `I-ORG`, `B-LOC`, `I-LOC`.

This dataset works with all recent versions of `datasets` / no tokens, no scripts, no errors.


In [19]:
# Load English split of WikiANN — parquet-native, always works
raw_datasets = load_dataset('unimelb-nlp/wikiann', 'en')
print(raw_datasets)

# Inspect one training sample
sample = raw_datasets['train'][0]
print('\nTokens  :', sample['tokens'])
print('NER tags:', sample['ner_tags'])


DatasetDict({
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 10000
    })
    train: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 20000
    })
})

Tokens  : ['R.H.', 'Saunders', '(', 'St.', 'Lawrence', 'River', ')', '(', '968', 'MW', ')']
NER tags: [3, 4, 0, 3, 4, 4, 0, 0, 0, 0, 0]


In [20]:
# Build label mappings
label_list = raw_datasets['train'].features['ner_tags'].feature.names
id2label   = {i: l for i, l in enumerate(label_list)}
label2id   = {l: i for i, l in enumerate(label_list)}
NUM_LABELS = len(label_list)
print(f'Labels ({NUM_LABELS}): {label_list}')

# Dataset split sizes
pd.DataFrame({
    'split'       : ['train', 'validation', 'test'],
    'num_examples': [len(raw_datasets[s]) for s in ['train', 'validation', 'test']],
})


Labels (7): ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']


,split,num_examples
0,train,20000
1,validation,10000
2,test,10000


In [21]:
# Drop 'langs' and 'spans' columns — not needed for training
cols_to_remove = [c for c in raw_datasets['train'].column_names
                  if c not in ('tokens', 'ner_tags')]
raw_datasets = raw_datasets.remove_columns(cols_to_remove)
print('Columns kept:', raw_datasets['train'].column_names)


Columns kept: ['tokens', 'ner_tags']


## 4. Preprocessing

**Sub-word alignment:** DistilBERT splits words into sub-tokens ( `Washington → ['Wash', '##ington']`).
- First sub-token → keeps the real NER label
- Continuation sub-tokens + special tokens → `-100` (ignored in loss & metrics)


In [22]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def tokenize_and_align_labels(examples):
    """Tokenise word lists and propagate NER labels to sub-word pieces."""
    tokenized = tokenizer(
        examples['tokens'],
        truncation=True,
        max_length=MAX_LENGTH,
        is_split_into_words=True,  # input is pre-tokenised word lists
    )
    all_labels = []
    for i, labels in enumerate(examples['ner_tags']):
        word_ids = tokenized.word_ids(batch_index=i)
        prev_wid, label_ids = None, []
        for wid in word_ids:
            if wid is None:          # [CLS] / [SEP] / [PAD]
                label_ids.append(-100)
            elif wid != prev_wid:    # first sub-token of a new word → real label
                label_ids.append(labels[wid])
            else:                    # continuation sub-token → ignore
                label_ids.append(-100)
            prev_wid = wid
        all_labels.append(label_ids)
    tokenized['labels'] = all_labels
    return tokenized

tokenized_datasets = raw_datasets.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=raw_datasets['train'].column_names,
)
print(tokenized_datasets)


config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

DatasetDict({
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 10000
    })
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 20000
    })
})


## 5. Feature Engineering

Contextual sub-word embeddings from DistilBERT are the features — no manual engineering needed.  
`DataCollatorForTokenClassification` applies **dynamic padding** per batch for memory efficiency.


In [23]:
data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer,
    padding=True,
    label_pad_token_id=-100,
)
print('Data collator ready — dynamic padding enabled.')


Data collator ready — dynamic padding enabled.


## 6. Model Training


In [24]:
# Load pre-trained DistilBERT with a token-classification head
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
)
model.to(DEVICE)
print(f'Parameters: {model.num_parameters():,}')


model.safetensors:   0%|          | 0.00/263M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-cased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Parameters: 65,196,295


In [25]:
def compute_metrics(eval_preds):
    """seqeval metrics: precision, recall, F1 over BIO label sequences."""
    logits, labels = eval_preds
    preds = np.argmax(logits, axis=-1)
    true_l, pred_l = [], []
    for p_row, l_row in zip(preds, labels):
        true_l.append([id2label[l] for l, p in zip(l_row, p_row) if l != -100])
        pred_l.append([id2label[p] for l, p in zip(l_row, p_row) if l != -100])
    return {
        'precision': precision_score(true_l, pred_l),
        'recall'   : recall_score(true_l, pred_l),
        'f1'       : f1_score(true_l, pred_l),
    }


In [28]:
training_args = TrainingArguments(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = NUM_EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    learning_rate               = LEARNING_RATE,
    weight_decay                = 0.01,
    eval_strategy         = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'f1',
    logging_steps               = 100,
    push_to_hub                 = False,
    report_to                   = 'none',
    seed                        = SEED,
    fp16                        = (DEVICE == 'cuda'),
)

trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = tokenized_datasets['train'],
    eval_dataset    = tokenized_datasets['validation'],
    processing_class      = tokenizer,
    data_collator   = data_collator,
    compute_metrics = compute_metrics,
)

train_result = trainer.train()
print(f"Runtime        : {train_result.metrics['train_runtime']:.1f}s")
print(f"Samples/second : {train_result.metrics['train_samples_per_second']:.1f}")

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Model saved → {OUTPUT_DIR}')


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.602438,0.552509,0.748075,0.791259,0.769061
2,0.453900,0.512345,0.774378,0.806630,0.790175
3,0.359110,0.513845,0.784287,0.814563,0.799138


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


Runtime        : 246.2s
Samples/second : 243.7


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved → ./ner_model


## 7. Evaluation


In [29]:
# Validation set metrics
val_metrics = trainer.evaluate(tokenized_datasets['validation'])
print('[Validation set]')
for k, v in val_metrics.items():
    print(f'  {k:<32} {v:.4f}' if isinstance(v, float) else f'  {k:<32} {v}')


[Validation set]
  eval_loss                        0.5138
  eval_precision                   0.7843
  eval_recall                      0.8146
  eval_f1                          0.7991
  eval_runtime                     13.3550
  eval_samples_per_second          748.7810
  eval_steps_per_second            23.4370
  epoch                            3.0000


In [30]:
# Full test-set seqeval report
test_preds = trainer.predict(tokenized_datasets['test'])
pred_ids   = np.argmax(test_preds.predictions, axis=-1)
label_ids  = test_preds.label_ids

true_l, pred_l = [], []
for p_row, l_row in zip(pred_ids, label_ids):
    true_l.append([id2label[l] for l, p in zip(l_row, p_row) if l != -100])
    pred_l.append([id2label[p] for l, p in zip(l_row, p_row) if l != -100])

print(classification_report(true_l, pred_l, digits=4))


              precision    recall  f1-score   support

         LOC     0.8033    0.8407    0.8215      4638
         ORG     0.6992    0.7191    0.7090      4745
         PER     0.8502    0.8880    0.8687      4518

   micro avg     0.7834    0.8145    0.7987     13901
   macro avg     0.7842    0.8159    0.7997     13901
weighted avg     0.7830    0.8145    0.7984     13901



## 8. Inference


In [32]:
# Build inference pipeline from the saved fine-tuned model
ner_pipe = pipeline(
    task                 = 'ner',
    model                = OUTPUT_DIR,
    tokenizer            = OUTPUT_DIR,
    aggregation_strategy = 'simple',   # merges B/I sub-word tokens
    device               = 0 if DEVICE == 'cuda' else -1,
)

news_articles = [
    ('Apple CEO Tim Cook announced a new partnership with OpenAI at WWDC '
     'in San Francisco. The deal will integrate ChatGPT into Siri.'),
    ('Manchester United signed Jude Bellingham from Borussia Dortmund. '
     'He will play under manager Erik ten Hag at Old Trafford.'),
    ('UN Secretary-General António Guterres warned about tensions between '
     'China and Taiwan at a session in New York.'),
]

for idx, text in enumerate(news_articles, 1):
    print(f'--- Article {idx} ---')
    for e in ner_pipe(text):
        print(f"  [{e['entity_group']:5s}] {e['word']:30s}  score={e['score']:.3f}")
    print()


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

--- Article 1 ---
  [ORG  ] Apple                           score=0.597
  [PER  ] Tim Cook                        score=0.983
  [ORG  ] OpenAI                          score=0.807
  [ORG  ] WWDC                            score=0.710
  [LOC  ] San Francisco                   score=0.949
  [ORG  ] Cha                             score=0.786
  [ORG  ] ##GPT                           score=0.721
  [ORG  ] Sir                             score=0.655

--- Article 2 ---
  [ORG  ] Manchester United               score=0.983
  [PER  ] Jude Bellingham                 score=0.958
  [ORG  ] Borussia Dortmund               score=0.939
  [PER  ] Erik ten Hag                    score=0.902
  [ORG  ] Old Trafford                    score=0.984

--- Article 3 ---
  [ORG  ] UN                              score=0.562
  [PER  ] Secretary                       score=0.516
  [PER  ] General                         score=0.644
  [PER  ] António Guterres                score=0.842
  [LOC  ] China           

In [33]:
# Structured entity extraction → pandas DataFrame
def extract_entities_df(text):
    return pd.DataFrame([{
        'entity_type': e['entity_group'],
        'entity_text': e['word'],
        'start'      : e['start'],
        'end'        : e['end'],
        'confidence' : round(e['score'], 4),
    } for e in ner_pipe(text)])

print('Article 1 — structured output:')
extract_entities_df(news_articles[0])


Article 1 — structured output:


,entity_type,entity_text,start,end,confidence
0,ORG,Apple,0,5,0.5972
1,PER,Tim Cook,10,18,0.9827
2,ORG,OpenAI,52,58,0.8068
3,ORG,WWDC,62,66,0.7104
4,LOC,San Francisco,70,83,0.9491
5,ORG,Cha,109,112,0.7859
6,ORG,##GPT,113,116,0.7210
7,ORG,Sir,122,125,0.6550


In [34]:
# spaCy en_core_web_sm baseline for side-by-side comparison
nlp_spacy = spacy.load('en_core_web_sm')
doc       = nlp_spacy(news_articles[0])

print('spaCy baseline (en_core_web_sm) — Article 1:')
pd.DataFrame([{
    'entity_type': ent.label_,
    'entity_text': ent.text,
    'start'      : ent.start_char,
    'end'        : ent.end_char,
} for ent in doc.ents])


spaCy baseline (en_core_web_sm) — Article 1:


,entity_type,entity_text,start,end
0,ORG,Apple,0,5
1,PERSON,Tim Cook,10,18
2,PERSON,OpenAI,52,58
3,GPE,San Francisco,70,83
4,GPE,Siri,122,126


In [35]:
# Batch inference + combined entity frequency summary
all_dfs  = [extract_entities_df(t) for t in news_articles]
combined = pd.concat(all_dfs, keys=[f'article_{i+1}' for i in range(len(all_dfs))])
combined.index.names = ['article', 'row']


print(combined.groupby('entity_type')['entity_text']
              .count().sort_values(ascending=False).to_string())
combined


Entity frequency across all articles:
entity_type
ORG    10
PER     6
LOC     4


entity_type        entity_text  start  end  confidence
article   row                                                       
article_1 0           ORG              Apple      0    5      0.5972
          1           PER           Tim Cook     10   18      0.9827
          2           ORG             OpenAI     52   58      0.8068
          3           ORG               WWDC     62   66      0.7104
          4           LOC      San Francisco     70   83      0.9491
          5           ORG                Cha    109  112      0.7859
          6           ORG              ##GPT    113  116      0.7210
          7           ORG                Sir    122  125      0.6550
article_2 0           ORG  Manchester United      0   17      0.9832
          1           PER    Jude Bellingham     25   40      0.9585
          2           ORG  Borussia Dortmund     46   63      0.9391
          3           PER       Erik ten Hag     92  104      0.9018
          4           ORG       Old Trafford    108  120      0.9835
article_3 0           ORG                 UN      0    2      0.5615
          1           PER          Secretary      3   12      0.5157
          2           PER            General     13   20      0.6444
          3           PER   António Guterres     21   37      0.8422
          4           LOC              China     68   73      0.6191
          5           LOC             Taiwan     78   84      0.7501
          6           LOC           New York    101  109      0.9761